In [1]:
import os
import sys
import re
import pandas as pd
import numpy as np
import json
from pathlib import Path
import random
import logging
import uuid

In [2]:
# load the data
poly_lib='extractions-from-paper/polymer_library.xlsx'
poly_names=pd.read_excel(poly_lib)
synthesis_file='extractions-from-paper/synthetic_procedures.xlsx'
synth=pd.read_excel(synthesis_file)
properties=pd.read_csv('extractions-from-paper/properties.csv')
properties.drop(index=0,inplace=True)
properties["Polymer Number"]=np.array(properties["Polymer Number"].values,dtype=int)
poly_names=poly_names.merge(properties,left_on="Polymer Number", right_on="Polymer Number")
poly_names=poly_names.merge(synth,left_on="Method Key", right_on="Label")

In [3]:
for i,x in enumerate(poly_names['Monomer 2'].values):
    if(not isinstance(x,str) or len(x)<3):
        poly_names['Monomer 2'].values[i]=np.nan

In [4]:
for i,x in enumerate(poly_names['Monomer 1: Monomer 2 (mole basis)'].values):
    if(not isinstance(x,str) or len(x)<3):
        poly_names['Monomer 2'].values[i]=np.nan

In [5]:
SOLUBILITY_TEXT = 'dichloromethane at a concentration of 40 mg/mL at room temperature'
DEGRADABILITY_TEXT = 'in a clear zone assay using Pseudomonas lemoignei as the microorganism'
PHYSICAL_STATE_TEXT = 'determined by visual observation and polarized microscope analysis'

def _biodeg_answer(record):
    return 'Biodegradable' if record['Biodegradability'] == 'yes' else 'Non-biodegradable'

def _solubility_answer(record):
    return 'soluble' if record['Solubility'] == 'yes' else 'insoluble'

def _is_homopolymer(record):
    """Use the Product field as the authoritative source for polymer classification."""
    product = record.get('Product', '')
    if isinstance(product, str):
        return 'homopolymer' in product.lower()
    # Fallback for M33/M34 or missing Product field
    return pd.isna(record['Monomer 2'])

In [6]:
def generate_unique_id(existing_ids):
    """
    Generate a new unique custom ID.
    Args:
        existing_ids (List[str]): A list of existing IDs to check against.
    Returns:
        str: A new unique ID.
    """
    while True:
        new_id = uuid.uuid4().hex
        if new_id not in existing_ids:
            return new_id

In [7]:
def data_record_to_context(record):
    """
    Convert a database record to context.
    Args:
        record (pd.Series): A row from the database.
    Returns:
        str: A context text.
    """
    if pd.isna(record['Monomer 2']):
        Synth = (f"{record['Polymer class']} was synthesised using {record['Type']} "
                 f"in {record['Medium']} using monomer {record['Monomer 1']}. "
                 f"BigSMILES of the polymer is {record['BigSMILES']}. ")
    else:
        Synth = (f"{record['Polymer class']} was synthesised using {record['Type']} "
                 f"in {record['Medium']} using monomers {record['Monomer 1']} and "
                 f"{record['Monomer 2']}. The mole ratio of monomer 1 to monomer 2 "
                 f"in the polymer is {record['Monomer 1: Monomer 2 (mole basis)']}. "
                 f"BigSMILES of the polymer is {record['BigSMILES']}. ")

    if record['Label'] in ('M33', 'M34'):
        Copol_homopol = ('Resulting polymer is a homopolymer. ' if pd.isna(record['Monomer 2'])
                         else 'Resulting polymer is a copolymer. ')
    else:
        Copol_homopol = ('Resulting polymer is a homopolymer. ' if pd.isna(record['Monomer 2'])
                         else f"Resulting polymer is {record['Product']}. ")

    if not pd.isna(record['Catalist']):
        Catalist=f"As catalist {record['Catalist']} was used"
    else:
        Catalist=''

    if record['Mn (kDa)'] > 0 and record['Mw (kDa)'] > 0:
        Mol_weight = (f"Number average molecular weight, Mn, of the polymer is "
                      f"{record['Mn (kDa)']} kDa. Weight average molecular weight, Mw, "
                      f"of the polymer is {record['Mw (kDa)']} kDa. ")
    else:
        Mol_weight = ''

    if record['D']>0:
        Polydispersity = f"Polydispersity is {record['D']}. "
    else:
        Polydispersity=''

    if record['Solubility'] == 'yes':
        Solubility = f'Resulting polymer is soluble in {SOLUBILITY_TEXT}. '
    elif record['Solubility'] == 'no':
        Solubility = f'Resulting polymer is insoluble in {SOLUBILITY_TEXT}. '
    else:
        Solubility = ''

    if not pd.isna(record['Physical State']):
        Physical_state = (f"Physical state is {record['Physical State']}. "
                          f"Physical state of the polymer {PHYSICAL_STATE_TEXT}. ")
    else:
        Physical_state=''

    if record['Biodegradability'] == 'yes':
        Degradability = f'Biodegradable {DEGRADABILITY_TEXT}.'
    elif record['Biodegradability'] == 'no':
        Degradability = f'Non-biodegradable {DEGRADABILITY_TEXT}.'
    else:
        Degradability = ''

    return Synth + Copol_homopol + Catalist + Mol_weight + Polydispersity + Solubility + Physical_state + Degradability

In [8]:
def _simple_qas(record):
    questions, answers = [], []

    questions.append("What monomers were used for polymer synthesis?")
    if pd.isna(record['Monomer 2']):
        answers.append(record['Monomer 1'])
    else:
        answers.append(f"{record['Monomer 1']} and {record['Monomer 2']}")
        questions.append("What was the Monomer 1 : Monomer 2 molar ratio?")
        answers.append(str(record['Monomer 1: Monomer 2 (mole basis)']))

    questions.append("What synthesis method was used?")
    answers.append(record['Type'])

    questions.append("Was the reaction conducted in bulk (melt), solution, or at a solution interface?")
    answers.append(record['Medium'])

    if not pd.isna(record['Mn (kDa)']):
        questions.append("What is the number average molecular weight (Mn) of the polymer?")
        answers.append(f"{record['Mn (kDa)']} kDa")

        questions.append(f"From which monomers was the polymer with Mn = {record['Mn (kDa)']} kDa synthesised?")
        if pd.isna(record['Monomer 2']):
            answers.append(record['Monomer 1'])
        else:
            answers.append(f"{record['Monomer 1']} and {record['Monomer 2']}")

        if not pd.isna(record['Biodegradability']):
            questions.append(f"Is the polymer with Mn = {record['Mn (kDa)']} kDa "
                             f"biodegradable {DEGRADABILITY_TEXT}?")
            answers.append(_biodeg_answer(record))
        
        if not pd.isna(record['Solubility']):
            questions.append(f"Is the polymer with Mn = {record['Mn (kDa)']} kDa "
                             f"soluble in {SOLUBILITY_TEXT}?")
            answers.append(_solubility_answer(record))

    if not pd.isna(record['Mw (kDa)']):
        questions.append("What is the weight average molecular weight (Mw) of the polymer (value without units)?")
        answers.append(f"{record['Mw (kDa)']} kDa")

    if not pd.isna(record['Biodegradability']):
        questions.append(f"Is this polymer biodegradable {DEGRADABILITY_TEXT}?")
        answers.append(_biodeg_answer(record))

    if not pd.isna(record['Solubility']):
        questions.append(f"Is this polymer soluble in {SOLUBILITY_TEXT}?")
        answers.append(_solubility_answer(record))

    if not pd.isna(record['Physical State']):
        questions.append(f"In which physical state is the polymer, {PHYSICAL_STATE_TEXT}?")
        answers.append(record['Physical State'])

    questions.append("What is the BigSMILES string for the polymer?")
    answers.append(record['BigSMILES'])

    return questions, answers

In [9]:
NAN_IMPOSSIBLE_TEMPLATES = [
    {
        'question': "What is the molar ratio of Monomer 1 to Monomer 2?",
        'condition': lambda r: pd.isna(r['Monomer 2']),
        'answer': "This information is not available in the context."
    },
    {
        'question': "What is the number average molecular weight (Mn) of the polymer?",
        'condition': lambda r: pd.isna(r['Mn (kDa)']),
        'answer': "This information is not available in the context."
    },
    {
        'question': "What polydispersity of the polymer?",
        'condition': lambda r: pd.isna(r['D']),
        'answer': "This information is not available in the context."
    },
    {
        'question': "What is the weight average molecular weight (Mw) of the polymer?",
        'condition': lambda r: pd.isna(r['Mw (kDa)']),
        'answer': "This information is not available in the context."
    },
    {
        'question': f"Is the polymer soluble in {SOLUBILITY_TEXT}?",
        'condition': lambda r: r['Solubility'] not in ('yes', 'no'),
        'answer': "This information is not available in the context."
    },
    {
        'question': f"Is the polymer biodegradable {DEGRADABILITY_TEXT}?",
        'condition': lambda r: r['Biodegradability'] not in ('yes', 'no'),
        'answer': "This information is not available in the context."
    },
    {
        'question': f"What is the physical state of the sample?",
        'condition': lambda r: pd.isna(r['Physical State']),
        'answer': "This information is not available in the context."
    },
]

In [10]:
def data_record_to_QAs(
    record,
    impossible_fraction: float = 0.4,
    random_seed: int = None,
):
    """
    Convert a database record to question-answer pairs.

    Args:
        record (pd.Series): A row from the polymer database.
        impossible_fraction (float): Fraction of total questions that should be
            impossible/unanswerable. 0.0 disables impossible questions entirely.
        random_seed (int): Optional seed for reproducibility.
    """
    if random_seed is not None:
        random.seed(random_seed)

    qa_dicts = []

    # Simple QAs
    simple_qs, simple_as = _simple_qas(record)
    for q, a in zip(simple_qs, simple_as):
        qa_dicts.append({
            'question': q, 'answer': a,
            'type': 'simple', 'sub_questions': [], 'answerable': True
        })

    # Build impossible pool
    impossible_pool = []

    for tmpl in NAN_IMPOSSIBLE_TEMPLATES:
        if tmpl['condition'](record):
            impossible_pool.append({
                'question': tmpl['question'],
                'answer': tmpl['answer'],
                'type': 'impossible_nan',
                'answerable': False
            })

    # Sample to hit target fraction
    n_answerable = len(qa_dicts)
    n_impossible_target = round(
            (impossible_fraction * n_answerable) / max(1 - impossible_fraction, 1e-9)
        )
    if len(impossible_pool) >= n_impossible_target:
        sampled = random.sample(impossible_pool, n_impossible_target)
    else:
        sampled = impossible_pool
    qa_dicts.extend(sampled)
    random.shuffle(qa_dicts)
    return qa_dicts

In [11]:
existing_ids=[]
data=[]
for i in range(len(poly_names)):
    record=poly_names.iloc[i]
    qas_record=[]
    context=data_record_to_context(record)
    qa_list=data_record_to_QAs(record)
    for item in qa_list:
        ind=generate_unique_id(existing_ids)
        existing_ids.append(ind)
        if(item['type']=='impossible_nan'):
            comment=item['answer']
            entry={
                    'id': ind,
                    'context': context,
                    'question': item['question'],
                    'is_impossible': True,
                     "answers": {
                                 "text": [],
                                 "answer_start": [],
                                },
                    'comment': comment,
                }
        elif(item['type']=='simple'):
            answer=item['answer']
            answer_start=context.find(answer)
            entry={
                    'id': ind,
                    'context': context,
                    'question': item['question'],
                    'is_impossible': False,
                     "answers": {
                                 "text": [answer],
                                 "answer_start": [answer_start],
                                },
                }
        qas_record.append(entry)
        
    data.append(qas_record)

In [12]:
len(data)

658

In [13]:
with open('qa_pairs_extractive_nomix.json','w') as file:
    json.dump(data,file)

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
train, test = train_test_split(data, test_size=0.1, random_state=42)
train, val = train_test_split(train, test_size=0.1, random_state=42)

In [16]:
# no mixing
train_data=[]
for record in train:
    for entry in record:
        single_entry={}
        single_entry['context']=entry['context']
        single_entry['question']=entry['question']
        single_entry['answers']=entry['answers']
        single_entry['is_impossible']=entry['is_impossible']
        train_data.append(single_entry)
random.shuffle(train_data)

In [17]:
val_data=[]
for record in val:
    for entry in record:
        single_entry={}
        single_entry['context']=entry['context']
        single_entry['question']=entry['question']
        single_entry['answers']=entry['answers']
        single_entry['is_impossible']=entry['is_impossible']
        val_data.append(single_entry)
random.shuffle(val_data)

In [18]:
test_data=[]
for record in test:
    for entry in record:
        single_entry={}
        single_entry['context']=entry['context']
        single_entry['question']=entry['question']
        single_entry['answers']=entry['answers']
        single_entry['is_impossible']=entry['is_impossible']
        test_data.append(single_entry)
random.shuffle(test_data)

In [19]:
with open('train_with_impossible.json','w') as file:
    json.dump(train_data,file)
with open('validation_with_impossible.json','w') as file:
    json.dump(val_data,file)
with open('test_with_impossible.json','w') as file:
    json.dump(test_data,file)